# Pandas Week 22 — Data Loading, Cleaning, Filtering & Grouping
**Name:** David Okusanya

This notebook covers:
1. Loading data into Pandas DataFrames from different formats (JSON and tab-separated text)
2. Cleaning the data (missing values, duplicates, inconsistent formatting)
3. Filtering and grouping to summarize the data

Two datasets are used:
- `json_sample.json` — a small movie ratings dataset (Rotten Tomatoes / IMDB style scores)
- `countries_of_the_world.txt` — a country-to-region mapping (tab-separated)

In [1]:
import pandas as pd
import numpy as np
import json

pd.set_option('display.max_columns', None)

## 1. Loading data from different formats

### 1a. JSON — movie ratings
The JSON file is a list containing a single dictionary, where each key is a movie title and each value is another dictionary of its stats. That's not a flat table, so it needs `from_dict` with `orient='index'` rather than a plain `read_json`.

In [2]:
with open('data/json_sample.json', 'r') as f:
    raw = json.load(f)

movies_df = pd.DataFrame.from_dict(raw[0], orient='index')
movies_df.index.name = 'Title'
movies_df = movies_df.reset_index()
movies_df.head()

,Title,Genre,Gross,IMDB Metascore,Popcorn Score,Rating,Tomato Score,popcornscore,rating,tomatoscore
0,12 Strong,Action,"$453,173",54,72,R,54,NaN,NaN,NaN
1,Den Of Thieves,Action,"$491,898",49,69,R,40,NaN,NaN,NaN
2,Fifty Shades Freed,Drama,unknown,34,unknown,unrated,unkown,NaN,NaN,NaN
3,Golden Exits,Drama,unknown,72,unknown,unrated,unkown,NaN,NaN,NaN
4,Hostiles,Adventure,"$548,886",65,71,R,72,NaN,NaN,NaN


### 1b. Tab-separated text — countries and regions
This file isn't a `.csv` in the strict sense (it's tab-delimited, not comma-delimited), so `sep='\t'` is passed to `read_csv`.

In [3]:
countries_df = pd.read_csv('data/countries_of_the_world.txt', sep='\t')
countries_df.head()

,Country,Region
0,Afghanistan,ASIA (EX. NEAR EAST)
1,Albania,EASTERN EUROPE
2,Algeria,NORTHERN AFRICA
3,American Samoa,OCEANIA
4,Andorra,WESTERN EUROPE


### 1c. Excel workbook — world population
The workbook has two sheets (`world_population` and `Sheet1`), so `sheet_name` is used to pick the one with the full data. `pd.ExcelFile` first lists what's inside before loading anything, which is a good habit for a workbook you haven't opened before.

In [4]:
xl = pd.ExcelFile('data/world_population_excel_workbook.xlsx')
print(xl.sheet_names)

pop_df = pd.read_excel(xl, sheet_name='world_population')
pop_df.head()

['world_population', 'Sheet1']


,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area (kmÂ²),Density (per kmÂ²),Growth Rate,World Population Percentage
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,1.0257,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.9957,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,1.0164,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.9831,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,1.0100,0.00


## 2. Cleaning the data

**Movies dataset issues to fix:**
- Column names aren't consistent — some rows use `Popcorn Score` / `Tomato Score` / `Rating`, others use lowercase `popcornscore` / `tomatoscore` / `rating`. These need to be merged into single columns.
- Missing values show up in different disguises: the string `"unknown"`, the misspelled `"unkown"`, `None`/`null`, and even a placeholder `-1`.
- `Gross` is stored as text with a `$` and commas (e.g. `"$453,173"`), so it can't be used for numeric analysis yet.
- Some titles are missing a `Genre` entirely.

**Countries dataset issues to fix:**
- Country and Region names have trailing/leading whitespace.
- Region names have inconsistent padding (extra spaces baked into the strings).

In [5]:
# --- Merge the inconsistent column name pairs in movies_df ---
movies_df['PopcornScore'] = movies_df['Popcorn Score'].combine_first(movies_df['popcornscore'])
movies_df['TomatoScore']  = movies_df['Tomato Score'].combine_first(movies_df['tomatoscore'])
movies_df['Rating']       = movies_df['Rating'].combine_first(movies_df['rating'])

movies_df = movies_df.drop(columns=['Popcorn Score', 'popcornscore', 'Tomato Score', 'tomatoscore', 'rating'])
movies_df.head()

,Title,Genre,Gross,IMDB Metascore,Rating,PopcornScore,TomatoScore
0,12 Strong,Action,"$453,173",54,R,72,54
1,Den Of Thieves,Action,"$491,898",49,R,69,40
2,Fifty Shades Freed,Drama,unknown,34,unrated,unknown,unkown
3,Golden Exits,Drama,unknown,72,unrated,unknown,unkown
4,Hostiles,Adventure,"$548,886",65,R,71,72


In [6]:
# --- Standardize all the "missing value" disguises to real NaN ---
missing_tokens = ['unknown', 'unkown', 'unrated', -1, None]
movies_df = movies_df.replace(missing_tokens, np.nan)

# IMDB Metascore was stored as text; convert to numeric now that junk values are gone
movies_df['IMDB Metascore'] = pd.to_numeric(movies_df['IMDB Metascore'], errors='coerce')
movies_df['TomatoScore']    = pd.to_numeric(movies_df['TomatoScore'], errors='coerce')
movies_df['PopcornScore']   = pd.to_numeric(movies_df['PopcornScore'], errors='coerce')

# Clean up Gross: strip $ and commas, convert to float
movies_df['Gross'] = (movies_df['Gross']
                       .astype(str)
                       .str.replace('[$,]', '', regex=True)
                       .replace('nan', np.nan)
                       .astype(float))

movies_df.isna().sum()

Title              0
Genre             22
Gross             28
IMDB Metascore    22
Rating             6
PopcornScore       7
TomatoScore        7
dtype: int64

In [7]:
# --- Drop duplicate rows if any exist ---
print('Duplicates before:', movies_df.duplicated().sum())
movies_df = movies_df.drop_duplicates()
print('Duplicates after:', movies_df.duplicated().sum())

movies_df.head()

Duplicates before: 0
Duplicates after: 0


,Title,Genre,Gross,IMDB Metascore,Rating,PopcornScore,TomatoScore
0,12 Strong,Action,453173.0,54.0,R,72.0,54.0
1,Den Of Thieves,Action,491898.0,49.0,R,69.0,40.0
2,Fifty Shades Freed,Drama,NaN,34.0,NaN,NaN,NaN
3,Golden Exits,Drama,NaN,72.0,NaN,NaN,NaN
4,Hostiles,Adventure,548886.0,65.0,R,71.0,72.0


In [8]:
# --- Clean up countries_df: strip whitespace from every text column ---
countries_df['Country'] = countries_df['Country'].str.strip()
countries_df['Region']  = countries_df['Region'].str.strip()

print('Duplicates:', countries_df.duplicated().sum())
countries_df = countries_df.drop_duplicates()
countries_df.head()

Duplicates: 0


,Country,Region
0,Afghanistan,ASIA (EX. NEAR EAST)
1,Albania,EASTERN EUROPE
2,Algeria,NORTHERN AFRICA
3,American Samoa,OCEANIA
4,Andorra,WESTERN EUROPE


**Population dataset:** no missing values or duplicate rows, but the `Area (km²)` and `Density (per km²)` column headers came in with a broken `²` character due to an encoding quirk in the source file — that's fixed below so the columns are usable by name.

In [9]:
pop_df = pop_df.rename(columns={
    'Area (kmÂ²)': 'Area (km2)',
    'Density (per kmÂ²)': 'Density (per km2)'
})

print('Missing values:', pop_df.isna().sum().sum(), 'total')
print('Duplicates:', pop_df.duplicated().sum())
pop_df.columns.tolist()

Missing values: 0 total
Duplicates: 0


['Rank',
 'CCA3',
 'Country',
 'Capital',
 'Continent',
 '2022 Population',
 '2020 Population',
 '2015 Population',
 '2010 Population',
 '2000 Population',
 '1990 Population',
 '1980 Population',
 '1970 Population',
 'Area (km2)',
 'Density (per km2)',
 'Growth Rate',
 'World Population Percentage']

## 3. Filtering and grouping

### Filtering
Example: movies rated **R** with a Tomato Score above 80.

In [10]:
high_rated_r = movies_df[(movies_df['Rating'] == 'R') & (movies_df['TomatoScore'] > 80)]
high_rated_r[['Title', 'Rating', 'TomatoScore', 'PopcornScore']]

,Title,Rating,TomatoScore,PopcornScore
14,The Shape Of Water,R,92.0,78.0
16,A Fantastic Woman (Una Mujer Fantástica),R,90.0,83.0
19,Call Me By Your Name,R,96.0,87.0
24,"I, Tonya",R,90.0,89.0
27,Molly'S Game,R,82.0,85.0
29,Phantom Thread,R,91.0,68.0
34,The Disaster Artist,R,91.0,89.0
35,The Insult (L'Insulte),R,89.0,86.0
36,"Three Billboards Outside Ebbing, Missouri",R,93.0,87.0


### Grouping — movies
Average Tomato/Popcorn score and total gross by Genre (only rows that have a Genre and Gross figure).

In [11]:
genre_summary = (movies_df.dropna(subset=['Genre'])
                  .groupby('Genre')
                  .agg(avg_tomato_score=('TomatoScore', 'mean'),
                       avg_popcorn_score=('PopcornScore', 'mean'),
                       total_gross=('Gross', 'sum'),
                       movie_count=('Title', 'count'))
                  .sort_values('avg_tomato_score', ascending=False))
genre_summary

,avg_tomato_score,avg_popcorn_score,total_gross,movie_count
Genre,,,,
Animation,100.000000,89.000000,184414.0,2
Adventure,82.000000,74.500000,997173.0,2
Action,53.250000,75.250000,2426401.0,4
Biography,51.666667,67.666667,1787262.0,3
Comedy,NaN,NaN,0.0,1
Drama,NaN,NaN,0.0,4


Count of movies per Rating category:

In [12]:
movies_df['Rating'].value_counts()

Rating
R       15
PG13    10
PG       5
NR       2
Name: count, dtype: int64

### Grouping — countries
Number of countries per Region.

In [13]:
region_counts = countries_df.groupby('Region')['Country'].count().sort_values(ascending=False)
region_counts

Region
SUB-SAHARAN AFRICA      51
LATIN AMER. & CARIB     45
ASIA (EX. NEAR EAST)    28
WESTERN EUROPE          28
OCEANIA                 21
NEAR EAST               16
C.W. OF IND. STATES     12
EASTERN EUROPE          12
NORTHERN AFRICA          6
NORTHERN AMERICA         5
BALTICS                  3
Name: Country, dtype: int64

### Filtering — population
Countries with a 2022 population over 100 million.

In [14]:
over_100m = pop_df[pop_df['2022 Population'] > 100_000_000][['Country', 'Continent', '2022 Population']]
over_100m.sort_values('2022 Population', ascending=False)

,Country,Continent,2022 Population
41,China,Asia,1425887337
92,India,Asia,1417173173
221,United States,North America,338289857
93,Indonesia,Asia,275501339
156,Pakistan,Asia,235824862
149,Nigeria,Africa,218541212
27,Brazil,South America,215313498
16,Bangladesh,Asia,171186372
171,Russia,Europe,144713314
131,Mexico,North America,127504125


### Grouping — population by continent
Total 2022 population and average growth rate per continent.

In [15]:
continent_summary = (pop_df.groupby('Continent')
                      .agg(total_2022_population=('2022 Population', 'sum'),
                           avg_growth_rate=('Growth Rate', 'mean'),
                           country_count=('Country', 'count'))
                      .sort_values('total_2022_population', ascending=False))
continent_summary

,total_2022_population,avg_growth_rate,country_count
Continent,,,
Asia,4721383274,1.009384,50
Africa,1426730932,1.021244,57
Europe,743147538,1.002256,50
North America,600296136,1.004175,40
South America,436816608,1.007957,14
Oceania,45038554,1.007383,23


## 4. Save the cleaned data

Exporting the cleaned DataFrames to CSV as a final check that everything loads back correctly.

In [16]:
movies_df.to_csv('movies_cleaned.csv', index=False)
countries_df.to_csv('countries_cleaned.csv', index=False)
pop_df.to_csv('population_cleaned.csv', index=False)
print('Saved cleaned files.')

Saved cleaned files.


## Notes
- `countries_of_the_world.csv` was part of the original assignment materials but wasn't readable in this environment, so it isn't used directly here. It isn't needed for the "different formats" requirement anyway — this notebook already loads from JSON, tab-delimited text, and Excel (three distinct formats).